# Texas Fiscal Radar: walkthrough

**Question:** after the Texas Legislature passes its bills, which state agencies face new costs and duties, how large are
they relative to what each agency already spends, and what is coming in the 2027 session?

**Data:** the Legislature's own files (bill histories and Legislative Budget Board fiscal notes, from its public FTP site)
and the Comptroller's FY2024 spending by agency. Everything below is read from the project's Azure SQL warehouse,
which the Data Factory → Synapse → SQL pipeline fills. Nothing here is hand-typed.

*Words used:* a **fiscal note** is the Legislative Budget Board's estimate of what a bill would cost; **General Revenue
(GR)** is the state's main checking account; **enacted** means the bill became law; **FY2026-27** is the two-year budget
period that started September 1, 2025.

In [1]:
import sys, pandas as pd
sys.path.insert(0, "../src")
from txfr.common import connect_sql, load_env
load_env(); conn = connect_sql()

def q(sql):
    cur = conn.cursor(); cur.execute(sql)
    df = pd.DataFrame.from_records([tuple(r) for r in cur.fetchall()], columns=[d[0] for d in cur.description])
    for c in df.columns:
        if df[c].dtype == object and df[c].dropna().map(lambda v: hasattr(v, "as_integer_ratio")).all():
            df[c] = df[c].astype(float)
    return df
pd.set_option("display.max_colwidth", 80); pd.set_option("display.float_format", "{:,.0f}".format)

## 1. Can we trust the warehouse?

Before any analysis: the Legislature publishes its own count of bills that passed both chambers in 2025 (619 House +
594 Senate = 1,213). If the warehouse disagrees, every number after this is suspect.

In [2]:
q("SELECT passed_89r_hb_sb AS warehouse, passed_89r_hb_sb_official AS official, bills, fiscal_notes, "
  "ai_cost_rows, ai_rows_needing_review, last_load_utc FROM dw.v_data_quality")

,warehouse,official,bills,fiscal_notes,ai_cost_rows,ai_rows_needing_review,last_load_utc
0,1213,1213,10635,7334,1012,529,2026-11-10 09:01:01


## 2. How far bills get

Most filed bills never become law. Knowing where they stop tells an agency where to spend analyst time.

In [3]:
q("SELECT session_code, bill_type, filed, referred, passed_one_chamber, passed_both, enacted, vetoed "
  "FROM dw.v_session_funnel WHERE bill_type IN ('HB','SB') ORDER BY session_code, bill_type")

,session_code,bill_type,filed,referred,passed_one_chamber,passed_both,enacted,vetoed
0,891,HB,298,10,0,0,0,0
1,891,SB,71,26,17,0,0,0
2,892,HB,311,166,20,11,11,0
3,892,SB,57,22,17,10,9,1
4,89R,HB,5644,5644,1158,619,607,12
5,89R,SB,3075,3053,984,594,580,14
6,90R,HB,900,0,0,0,0,0
7,90R,SB,279,0,0,0,0,0


![funnel](../docs/img/funnel_89r.png)

## 3. What the enacted bills cost, by agency

A fiscal note gives one cost for the whole bill. When a bill names one agency, the cost is that agency's (**direct**).
When it names several, the AI step reads the note's explanation and extracts the amounts it states per agency; a row
counts only if its quote is in the note, its number is in the quote and its agency is on the note's list
(**AI-attributed**). Costs on multi-agency bills with no such statement stay **shared**: counted, never guessed.

In [4]:
q("SELECT TOP 12 agency_name, enacted_bills, direct_cost_2yr, ai_cost_2yr, shared_bills, new_fte, fy2024_spend "
  "FROM dw.v_agency_exposure ORDER BY attributed_cost_2yr DESC")

,agency_name,enacted_bills,direct_cost_2yr,ai_cost_2yr,shared_bills,new_fte,fy2024_spend
0,TEXAS EDUCATION AGENCY,84,"507,211,624","397,249,027",41,5,"42,382,151,853"
1,COMPTROLLER OF PUBLIC ACCOUNTS,306,"199,958,000","105,025,030",219,12,"405,648,077"
2,DEPARTMENT OF INFORMATION RESOURCES,31,"146,857,011","7,862,742",24,139,"776,958,714"
3,HEALTH AND HUMAN SERVICES COMMISSION,155,"39,478,715","85,529,250",105,8,"62,659,367,693"
4,TEACHER RETIREMENT SYSTEM OF TEXAS,15,0,"61,400,000",13,NaN,"14,249,264,062"
5,EMPLOYEES RETIREMENT SYSTEM OF TEXAS,35,0,"26,400,000",30,NaN,"2,634,265,055"
6,OFFICE OF COURT ADMINISTRATION,250,0,"21,000,295",162,NaN,"119,752,352"
7,OFFICE OF THE ATTORNEY GENERAL,99,0,"15,131,524",75,NaN,"5,489,191,973"
8,TEXAS WORKFORCE COMMISSION,48,0,"13,629,633",31,NaN,"5,693,720,295"
9,PUBLIC UTILITY COMMISSION OF TEXAS,37,"8,496,396","1,513,943",13,28,"59,500,661"


![exposure](../docs/img/agency_exposure.png)

## 4. Readiness: who should prepare first

Raw dollars favour big agencies. Dividing the new yearly cost by the agency's FY2024 spending shows where the change
is large *for that agency*. FY2024 predates these laws, so this is a yardstick of size, not a budget.

In [5]:
q("SELECT TOP 12 agency_name, attributed_cost_2yr, fy2024_spend, load_pct_of_spend, new_fte "
  "FROM dw.v_agency_exposure WHERE fy2024_spend > 1000000 ORDER BY load_pct_of_spend DESC")

,agency_name,attributed_cost_2yr,fy2024_spend,load_pct_of_spend,new_fte
0,STATE COMMISSION ON JUDICIAL CONDUCT,"3,861,261","1,470,840",131,NaN
1,COMPTROLLER OF PUBLIC ACCOUNTS,"304,983,030","405,648,077",38,12
2,DEPARTMENT OF INFORMATION RESOURCES,"154,719,753","776,958,714",10,139
3,TEXAS ANIMAL HEALTH COMMISSION,"3,890,748","22,117,928",9,5
4,OFFICE OF COURT ADMINISTRATION,"21,000,295","119,752,352",9,NaN
5,SUNSET ADVISORY COMMISSION,"623,388","3,608,209",9,NaN
6,PUBLIC UTILITY COMMISSION OF TEXAS,"10,010,339","59,500,661",8,28
7,TEXAS ETHICS COMMISSION,"595,240","4,034,529",7,NaN
8,STATE OFFICE OF ADMINISTRATIVE HEARINGS,"1,592,904","13,877,011",6,NaN
9,TEXAS BOND REVIEW BOARD,"130,000","1,307,561",5,NaN


![readiness](../docs/img/readiness.png)

## 5. Costs that grow after the budget period

Some laws are cheap in 2026-27 and expensive later. These belong in the next budget request.

In [6]:
q("SELECT TOP 10 bill_id, LEFT(caption, 70) AS caption, cost_2yr, gr_cost_outyears FROM dw.v_bill_cost "
  "WHERE enacted = 1 AND grows_later = 1 ORDER BY gr_cost_outyears DESC")

,bill_id,caption,cost_2yr,gr_cost_outyears
0,89R:SB2206,"Relating to a franchise tax credit for, and the application of sales a",0,"1,768,100,000"
1,89R:SB1379,Relating to increasing the criminal penalties for forgery.,"-158,250","57,871,909"
2,89R:SB1490,Relating to adult high school charter program funding.,"20,408,072","56,540,292"
3,89R:HB2517,Relating to the applicability of premium and maintenance taxes to the,0,"54,528,000"
4,89R:SB1021,Relating to changing the eligibility for community supervision of a pe,"-135,428","25,654,238"
5,892:HB16,Relating to the operation and administration of and practices and proc,"5,939,324","14,773,720"
6,89R:HB2313,Relating to the authority of certain municipalities to use certain tax,0,"9,300,000"
7,89R:SB1398,Relating to certain procedures in a suit affecting the parent-child re,"541,683","8,326,037"
8,89R:HB136,Relating to Medicaid coverage and reimbursement for lactation consulta,"6,188,382","5,090,805"


## 6. How much estimates move

Each version of a bill gets a new fiscal note. Bills whose estimate swung a lot deserve a contingency.

In [7]:
q("SELECT TOP 10 bill_id, status, first_version, first_cost_2yr, latest_version, latest_cost_2yr, change_pct "
  "FROM dw.v_estimate_drift WHERE session_code = '89R' ORDER BY ABS(change_2yr) DESC")

,bill_id,status,first_version,first_cost_2yr,latest_version,latest_cost_2yr,change_pct
0,89R:HB500,enacted,I,"10,053,768,569",F,"13,187,220,532",-31
1,89R:SB4,enacted,I,"1,035,602,146",F,"2,743,461,146",-165
2,89R:HB1868,enacted,I,"1,105,465,895",E,-0,100
3,89R:HB2,enacted,I,"7,568,572,060",F,"8,381,506,751",-11
4,89R:SB1750,passed one chamber,I,"663,082,476",S,"193,206,292",71
5,89R:HB9,enacted,I,"566,354,000",F,"193,504,000",66
6,89R:HB2249,passed one chamber,I,"401,710,359",H,"43,671,808",89
7,89R:SB568,enacted,I,"397,534,596",F,"699,778,662",-76
8,89R:HB120,enacted,I,"495,810,541",F,"213,466,252",57
9,89R:HB2825,in committee,I,"229,000,000",H,-0,100


![drift](../docs/img/estimate_drift.png)

## 7. The 2027 session, live

Prefiling for the 2027 session opened on 2026-11-09. The pipeline picks up new filings every morning from the
Legislature's change list. Bills whose subjects name an agency are an early watchlist, before any fiscal note exists.

In [8]:
q("SELECT agency_subject, COUNT(DISTINCT bill_id) AS bills FROM dw.v_watchlist WHERE agency_subject IS NOT NULL "
  "GROUP BY agency_subject ORDER BY bills DESC OFFSET 0 ROWS FETCH NEXT 12 ROWS ONLY")

,agency_subject,bills
0,HEALTH & HUMAN SERVICES COMMISSION,48
1,COMPTROLLER OF PUBLIC ACCOUNTS,28
2,"EDUCATION AGENCY, TEXAS",24
3,"EDUCATION, COMMISSIONER OF",21
4,"HIGHER EDUCATION COORDINATING BOARD, TEXAS",19
5,"ENVIRONMENTAL QUALITY, TEXAS COMMISSION ON",16
6,SECRETARY OF STATE,15
7,"WORKFORCE COMMISSION, TEXAS",15
8,"FAMILY & PROTECTIVE SERVICES, DEPARTMENT OF",13
9,"PUBLIC SAFETY, DEPARTMENT OF",13


![pace](../docs/img/filing_pace.png)

## 8. What the AI step did, and how it was checked

Every AI row is checked against the note itself, not sampled. Rows that fail any check wait for a person on the
dashboard's *Trust & review* page.

In [9]:
q("SELECT COUNT(*) AS rows, SUM(CASE WHEN needs_review = 0 THEN 1 ELSE 0 END) AS verified, "
  "SUM(CASE WHEN needs_review = 1 THEN 1 ELSE 0 END) AS needs_review, COUNT(DISTINCT bill_id) AS bills, "
  "COUNT(DISTINCT CASE WHEN reconciles = 1 THEN bill_id END) AS bills_complete_split FROM dw.fact_agency_cost_ai")

,rows,verified,needs_review,bills,bills_complete_split
0,1012,483,529,119,69


**Caveats.** Fiscal notes estimate one version of a bill; they are not appropriations. GR and all-funds figures
differ (each table says which). The AI split is a lower bound where a note's text doesn't itemise the whole cost.